---
## Setup

We use MPQP's high-level API to define circuits and IBM's Aer simulator to execute them. Run the cell below to load everything we need.

In [39]:
from mpqp import QCircuit
from mpqp.gates import H, X, Z, CNOT, TOF, CustomGate, CustomControlledGate
from mpqp.measures import BasisMeasure
from mpqp.execution import run
from mpqp.execution.devices import IBMDevice
import numpy as np

---
## Part 1 -- Bell States

The four **Bell states** form a maximally entangled basis for two qubits:

| State | Expression |
|---|---|
| $\lvert\Phi^+\rangle$ | $\frac{1}{\sqrt{2}}(\lvert00\rangle + \lvert11\rangle)$ |
| $\lvert\Phi^-\rangle$ | $\frac{1}{\sqrt{2}}(\lvert00\rangle - \lvert11\rangle)$ |
| $\lvert\Psi^+\rangle$ | $\frac{1}{\sqrt{2}}(\lvert01\rangle + \lvert10\rangle)$ |
| $\lvert\Psi^-\rangle$ | $\frac{1}{\sqrt{2}}(\lvert01\rangle - \lvert10\rangle)$ |

All four share the same skeleton: **H on qubit 0, then CNOT(0, 1)**. The difference is which single-qubit gates we apply *after* the CNOT:

- $\lvert\Phi^+\rangle$: nothing extra
- $\lvert\Phi^-\rangle$: apply $Z$ on qubit 0
- $\lvert\Psi^+\rangle$: apply $X$ on qubit 0
- $\lvert\Psi^-\rangle$: apply $X$ then $Z$ on qubit 0

### 1.1 -- Building $\lvert\Phi^+\rangle$ step by step

Let's start with the simplest Bell state. We need just two gates:

1. **Hadamard** on qubit 0 -- puts it into an equal superposition of $\lvert0\rangle$ and $\lvert1\rangle$.
2. **CNOT(0, 1)** -- entangles qubit 1 with qubit 0. If qubit 0 is $\lvert0\rangle$, qubit 1 stays $\lvert0\rangle$; if qubit 0 is $\lvert1\rangle$, qubit 1 flips to $\lvert1\rangle$.

The combined effect: the two qubits are always measured in the *same* state, but which state that is is completely random.

In [40]:
# Build the |Phi+> Bell state circuit
phi_plus = QCircuit(
    [H(0), CNOT(0, 1), BasisMeasure([0, 1], shots=1024)],
    nb_qubits=2,
    label="Bell State |Phi+>",
)
print(phi_plus)

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


Now let's simulate it. We expect to see counts *only* for $\lvert00\rangle$ and $\lvert11\rangle$, roughly 50/50.

In [41]:
result = run(phi_plus, IBMDevice.AER_SIMULATOR)
print(f"Counts:        {result.counts}")
print(f"Probabilities: {result.probabilities}")

Counts:        [518, 0, 0, 506]
Probabilities: [0.50585938 0.         0.         0.49414062]


**Check your understanding:**
- The counts array is indexed as $[\lvert00\rangle, \lvert01\rangle, \lvert10\rangle, \lvert11\rangle]$. Confirm that $\lvert01\rangle$ and $\lvert10\rangle$ have zero counts. Why must that be the case for $\lvert\Phi^+\rangle$?
- The probabilities don't land at exactly 0.5 / 0.5. What would happen if you increased `shots` to 100,000?

### 1.2 -- All four Bell states

We can generalize the circuit with a helper that selects which post-CNOT gates to apply.

In [42]:
def bell_state(name: str):
    """Prepare and measure a Bell state.

    Args:
        name: one of 'phi+', 'phi-', 'psi+', 'psi-'
    """
    post_cnot = {
        "phi+": [],
        "phi-": [Z(0)],
        "psi+": [X(0)],
        "psi-": [X(0), Z(0)],
    }
    assert name in post_cnot, f"Must be one of {list(post_cnot.keys())}"

    gates = [H(0), CNOT(0, 1)] + post_cnot[name] + [BasisMeasure([0, 1], shots=1024)]
    circuit = QCircuit(gates, nb_qubits=2, label=f"Bell State |{name}>")
    print(circuit)

    result = run(circuit, IBMDevice.AER_SIMULATOR)
    print(f"State: |{name}>")
    print(f"Counts:        {result.counts}")
    print(f"Probabilities: {result.probabilities}\n")
    return result

In [43]:
for name in ["phi+", "phi-", "psi+", "psi-"]:
    bell_state(name)

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
State: |phi+>
Counts:        [526, 0, 0, 498]
Probabilities: [0.51367188 0.         0.         0.48632812]

     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├┤M├
     └───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├─┤M├──╫─
          └───┘ └╥┘  ║ 
c: 2/════════════╩═══╩═
                 1   0 
State: |phi->
Counts:        [538, 0, 0, 486]
Probabilities: [0.52539062 0.         0.         0.47460938]

     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ X ├┤M├
     └───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├─┤M├──╫─
          └───┘ └╥┘  ║ 
c: 2/════════════╩═══╩═
                 1   0 
State: |psi+>
Counts:        [0, 495, 529, 0]
Probabilities: [0.         0.48339844 0.51660156 0.        ]

     ┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ X ├┤ Z ├┤M├
     └───┘┌─┴─┐└┬─┬┘└───┘└╥┘
q_1: ─────┤ X ├─┤M├───────╫─
          └───┘ └╥┘       ║ 
c: 2/════════════╩════════╩═
            

### 1.3 -- Exercise

1. Look at the results for $\lvert\Psi^+\rangle$ and $\lvert\Psi^-\rangle$. Which basis states have non-zero counts? How does this differ from the $\Phi$ states?
2. $\lvert\Phi^+\rangle$ and $\lvert\Phi^-\rangle$ produce the *same* measurement statistics (both give 50/50 on $\lvert00\rangle$ and $\lvert11\rangle$). If measurement alone can't distinguish them, what *can*? (Hint: think about the relative phase between the two terms.)

### 2.4 -- Exercise

1. For teleporting $\lvert0\rangle$, all four branches show ~50/50 counts. Does that mean teleportation failed? Explain why the counts look like this even though the protocol is correct. (Hint: we are not post-selecting on Alice's measurement.)
2. The no-cloning theorem says you cannot copy an arbitrary quantum state. How does teleportation avoid violating it? What happens to Alice's original qubit after the protocol?
3. **Bonus:** Modify the function to teleport $\lvert+\rangle = H\lvert0\rangle$ instead of a computational basis state. What would you change in the state preparation step?

In [44]:
# Example matrix: Hadamard matrix (det = -1)
M = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

psi = M[:, 0]  # first column
phi = M[:, 1]  # second column

print(f"psi = {psi}")
print(f"phi = {phi}")
print(f"Classical det = {np.linalg.det(M):.4f}")

psi = [0.70710678+0.j 0.70710678+0.j]
phi = [ 0.70710678+0.j -0.70710678+0.j]
Classical det = -1.0000+0.0000j


In [45]:
# Step 1: Build V (state prep unitary) and its conjugate transpose
#   V|0> = |phi>,  so  V = [[phi_0, -conj(phi_1)], [phi_1, conj(phi_0)]]
V = np.array([
    [phi[0], -np.conj(phi[1])],
    [phi[1],  np.conj(phi[0])],
])
V_dag = V.conj().T

# Step 2: Antisymmetric tensor (Levi-Civita in 2D)
epsilon = np.array([[0, 1], [-1, 0]], dtype=complex)

# Step 3: Combined operation U = epsilon @ V_dag
U = epsilon @ V_dag

print("V =")
print(np.round(V, 4))
print(f"\nU = ε @ V_dag =")
print(np.round(U, 4))

# Verify U is unitary
print(f"\nU @ U_dag = I? {np.allclose(U @ U.conj().T, np.eye(2))}")

V =
[[ 0.7071+0.j  0.7071+0.j]
 [-0.7071+0.j  0.7071-0.j]]

U = ε @ V_dag =
[[ 0.7071+0.j  0.7071+0.j]
 [-0.7071+0.j  0.7071+0.j]]

U @ U_dag = I? True


In [46]:
# Step 4: Build the Hadamard test circuit
#   Qubit 0 = data (initialized to |psi>)
#   Qubit 1 = ancilla

# Initialize |psi> on qubit 0, |0> on qubit 1
init_state = np.kron(psi, np.array([1, 0], dtype=complex))
circ = QCircuit.initializer(init_state)

# Hadamard test: H on ancilla, controlled-U, H on ancilla, measure ancilla
U_gate = CustomGate(U, [0], label="eps.V_dag")
CU_gate = CustomControlledGate([1], U_gate, label="C-U")

circ.add([H(1), CU_gate, H(1)])
circ.add(BasisMeasure([1], shots=2048))

print(circ)

     ┌────────────┐┌─────────┐        
q_0: ┤ U(π/2,0,0) ├┤ Unitary ├────────
     └───┬───┬────┘└────┬────┘┌───┐┌─┐
q_1: ────┤ H ├──────────■─────┤ H ├┤M├
         └───┘                └───┘└╥┘
c: 1/═══════════════════════════════╩═
                                    0 


In [47]:
# Step 5: Run and extract the determinant
result = run(circ, IBMDevice.AER_SIMULATOR)

counts = result.counts
shots = sum(counts)
p0 = counts[0] / shots  # P(ancilla = |0>)

det_quantum = 2 * p0 - 1
det_classical = np.linalg.det(M).real

print(f"Counts [|0>, |1>]: {counts}")
print(f"P(0) = {p0:.4f}")
print(f"Quantum  det = 2*P(0) - 1 = {det_quantum:.4f}")
print(f"Classical det             = {det_classical:.4f}")

Counts [|0>, |1>]: [1763, 285]
P(0) = 0.8608
Quantum  det = 2*P(0) - 1 = 0.7217
Classical det             = -1.0000


### 3B.4 -- General function

Let's wrap the above into a reusable function and test it on several matrices. Note that the columns must be **unit vectors** (the matrix need not be unitary, but each column must have norm 1 so they can be encoded as quantum states).

In [48]:
def quantum_determinant(mat: np.ndarray, shots: int = 4096) -> float:
    """Estimate the determinant of a 2x2 real matrix with unit-norm columns
    using a Hadamard test circuit.

    Args:
        mat: 2x2 array whose columns are unit vectors.
        shots: number of simulator shots (more shots = better precision).

    Returns:
        Estimated determinant (real-valued).
    """
    psi, phi = mat[:, 0].astype(complex), mat[:, 1].astype(complex)

    # V: state-prep unitary for phi  (V|0> = |phi>)
    V = np.array([
        [phi[0], -np.conj(phi[1])],
        [phi[1],  np.conj(phi[0])],
    ])

    # U = epsilon @ V_dag
    epsilon = np.array([[0, 1], [-1, 0]], dtype=complex)
    U = epsilon @ V.conj().T

    # Initialize qubit 0 to |psi>, qubit 1 to |0>
    init_state = np.kron(psi, np.array([1, 0], dtype=complex))
    circ = QCircuit.initializer(init_state)

    # Hadamard test on ancilla (qubit 1)
    U_gate = CustomGate(U, [0], label="eps.V_dag")
    CU_gate = CustomControlledGate([1], U_gate, label="C-U")
    circ.add([H(1), CU_gate, H(1)])
    circ.add(BasisMeasure([1], shots=shots))

    # Run
    result = run(circ, IBMDevice.AER_SIMULATOR)
    counts = result.counts
    p0 = counts[0] / sum(counts)

    return 2 * p0 - 1

Let's test on several matrices with known determinants.

In [49]:
test_matrices = {
    "Identity": np.eye(2),
    "Hadamard": np.array([[1, 1], [1, -1]]) / np.sqrt(2),
    "Pauli X": np.array([[0, 1], [1, 0]], dtype=float),
    "Rotation(pi/3)": np.array([
        [ np.cos(np.pi/3), np.sin(np.pi/3)],
        [-np.sin(np.pi/3), np.cos(np.pi/3)],
    ]),
}

print(f"{'Matrix':<16} {'Classical':>10} {'Quantum':>10}")
print("-" * 40)
for name, mat in test_matrices.items():
    det_q = quantum_determinant(mat, shots=8192)
    det_c = np.linalg.det(mat).real
    print(f"{name:<16} {det_c:>10.4f} {det_q:>10.4f}")

Matrix            Classical    Quantum
----------------------------------------
Identity             1.0000    -1.0000
Hadamard            -1.0000     0.7029
Pauli X             -1.0000     0.0193
Rotation(pi/3)       1.0000    -0.5005


### 3B.5 -- Exercise

1. The quantum estimate has shot noise. Run `quantum_determinant` on the identity matrix with `shots=256`, `shots=4096`, and `shots=65536`. How does the error scale with the number of shots? (Hint: think about the standard deviation of a binomial proportion.)
2. The Pauli X matrix $\begin{pmatrix}0 & 1\\1 & 0\end{pmatrix}$ has $\det = -1$. Its columns are $\lvert 1\rangle$ and $\lvert 0\rangle$. Can you predict $P(0)$ on the ancilla *without* running the circuit?
3. This method requires the columns to be unit vectors. If you have a general matrix $A$ with non-unit columns, how could you normalize it and recover $\det(A)$ from $\det(\hat{A})$? (Hint: $\det(\hat{A}) = \det(A) / (\|a_1\| \cdot \|a_2\|)$.)

---
## Summary

| Algorithm | Qubits | Key idea | What we verified |
|-----------|--------|----------|------------------|
| Bell states | 2 | H + CNOT creates maximal entanglement | Correlated 50/50 outcomes, zero probability on forbidden states |
| Teleportation | 3 | Shared entanglement + 2 classical bits transfer a qubit state | Correct state recovery across all 4 correction branches |
| Grover search (3A) | 2 | Oracle phase-flip + diffusion amplification | 100% success in a single iteration for $N = 4$ |
| Quantum determinant (3B) | 2 | Hadamard test extracts $\text{Re}\langle\psi\|U\|\psi\rangle$ | Quantum estimate matches $\det(M)$ for several test matrices |

These protocols illustrate the core phenomena that make quantum computing powerful: **entanglement**, **superposition**, and **interference** -- and show how quantum circuits can compute both quantum (teleportation, search) and classical (determinant) quantities.